# 01 — Kiểm tra dataset Reddit Data Science Posts

Notebook này tải dataset đã chốt từ Kaggle và tạo báo cáo audit ban đầu. Chạy trên Google Colab. Không chạy sentiment/topic trước khi xác nhận schema, license và khoảng thời gian.

Kaggle handle: `maksymshkliarevskyi/reddit-data-science-posts`. KaggleHub có thể dùng token từ Colab Secret `KAGGLE_API_TOKEN` nếu dataset yêu cầu xác thực. Không ghi token vào notebook.


In [ ]:
%pip -q install kagglehub pandas pyarrow


In [ ]:
from pathlib import Path
import json
import hashlib
import pandas as pd
import kagglehub
from google.colab import drive

drive.mount("/content/drive")
DATASET_HANDLE = "maksymshkliarevskyi/reddit-data-science-posts"
PROJECT_DIR = Path("/content/drive/MyDrive/RedditProject")
RAW_DIR = PROJECT_DIR / "data/raw/reddit-data-science-posts"
RAW_DIR.mkdir(parents=True, exist_ok=True)

dataset_dir = Path(kagglehub.dataset_download(DATASET_HANDLE, output_dir=str(RAW_DIR)))
print("Dataset path:", dataset_dir)


In [ ]:
# Liệt kê file dữ liệu và kích thước để ghi lại phiên bản đầu vào
files = sorted(p for p in dataset_dir.rglob("*") if p.is_file())
for p in files:
    print(f"{p.relative_to(dataset_dir)}\t{p.stat().st_size:,} bytes")

data_files = [p for p in files if p.suffix.lower() in {".csv", ".tsv", ".parquet"}]
if not data_files:
    raise FileNotFoundError("Không thấy CSV/TSV/Parquet; kiểm tra file đã tải và cập nhật parser.")


In [ ]:
# Đọc từng file theo chunks để audit mà không nạp toàn bộ dữ liệu vào RAM
import csv

def read_chunks(path):
    suffix = path.suffix.lower()
    if suffix == ".parquet":
        # Parquet được đọc theo file; nếu file quá lớn, thay bằng pyarrow.dataset scanner.
        yield pd.read_parquet(path)
    else:
        sep = "\t" if suffix == ".tsv" else ","
        yield from pd.read_csv(path, sep=sep, chunksize=100_000, low_memory=False)

summary = []
subreddit_counts = {}
for path in data_files:
    rows = 0
    missing = {}
    time_min = None
    time_max = None
    columns = None
    for chunk in read_chunks(path):
        columns = list(chunk.columns)
        rows += len(chunk)
        for col, n in chunk.isna().sum().items():
            missing[col] = missing.get(col, 0) + int(n)
        time_col = next((c for c in ("created_timestamp", "created_date", "created_utc") if c in chunk.columns), None)
        if time_col:
            values = chunk[time_col]
            numeric = pd.to_numeric(values, errors="coerce")
            if numeric.notna().mean() > 0.8:
                median_abs = numeric.dropna().abs().median()
                unit = "ms" if median_abs > 1e12 else "s" if median_abs > 1e9 else None
                parsed = pd.to_datetime(numeric, errors="coerce", utc=True, unit=unit)
            else:
                parsed = pd.to_datetime(values, errors="coerce", utc=True)
            lo, hi = parsed.min(), parsed.max()
            if pd.notna(lo): time_min = lo if time_min is None else min(time_min, lo)
            if pd.notna(hi): time_max = hi if time_max is None else max(time_max, hi)
        sub_col = next((c for c in ("subreddit", "subreddit_name") if c in chunk.columns), None)
        if sub_col:
            counts = chunk[sub_col].fillna("<NA>").astype(str).value_counts()
            for name, n in counts.items():
                subreddit_counts[name] = subreddit_counts.get(name, 0) + int(n)
    summary.append({"file": str(path.relative_to(dataset_dir)), "rows": rows, "columns": columns, "missing": missing, "time_min_utc": str(time_min), "time_max_utc": str(time_max)})

print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\nSố subreddit:", len(subreddit_counts))
print(pd.Series(subreddit_counts, name="rows").sort_values(ascending=False).to_string())


In [ ]:
# Ghi manifest và checksum để truy vết bản dữ liệu đã tải
def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()

manifest = {
    "dataset_handle": DATASET_HANDLE,
    "files": [{
        "path": str(p.relative_to(dataset_dir)),
        "size_bytes": p.stat().st_size,
        "sha256": sha256_file(p),
    } for p in data_files],
    "audit": summary,
}
manifest_path = RAW_DIR / "dataset_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("Manifest saved to:", manifest_path)


## Sau khi chạy

Ghi nhận từ output: phiên bản/nguồn dữ liệu, danh sách file, số dòng, schema, missing, phạm vi timestamp và số subreddit. Kiểm tra trang Kaggle để ghi license/điều khoản sử dụng. Nếu timestamp không parse được hoặc tên cột khác dự kiến, cập nhật ETL trước khi tiếp tục. File manifest hiện nằm trong runtime Colab; tải về hoặc lưu vào Drive để giữ lại sau khi phiên làm việc kết thúc.
